# dring data gallery

This notebook scans `/media/lhyang/SHAO_HPC/project_papper2/dring/data` and visualizes every packaged data product. Radial intensity profiles are plotted with shaded RMS errors; 2D opacity arrays are plotted with color bars.

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt

DATA = Path('/media/lhyang/SHAO_HPC/project_papper2/dring/data')
print(DATA)
print('\n'.join(str(p.relative_to(DATA)) for p in sorted(DATA.rglob('*')) if p.is_file()))

## Radial intensity profiles

Each `*_band*.npz` file is one wavelength band with `radius`, `intensity`, and `rms_error`.

In [ ]:
band_files = sorted(DATA.glob('*_band*.npz'))
groups = {}
for path in band_files:
    group = path.stem.split('_band')[0]
    groups.setdefault(group, []).append(path)

print(f'{len(band_files)} band files in {len(groups)} groups')
for group, paths in groups.items():
    print(f'{group}: {len(paths)} bands')

In [ ]:
for group, paths in groups.items():
    paths = sorted(paths)
    cmap = plt.get_cmap('viridis')
    colors = cmap(np.linspace(0.08, 0.92, len(paths))) if len(paths) > 1 else [cmap(0.5)]

    fig, ax = plt.subplots(figsize=(8.2, 4.8), dpi=150)
    all_positive = []
    for color, path in zip(colors, paths):
        z = np.load(path)
        radius = np.asarray(z['radius'], dtype=float)
        intensity = np.asarray(z['intensity'], dtype=float)
        rms_error = np.asarray(z['rms_error'], dtype=float)
        finite = np.isfinite(radius) & np.isfinite(intensity) & np.isfinite(rms_error)
        positive = finite & (intensity > 0)
        if np.any(positive):
            all_positive.append(intensity[positive])
            floor = np.nanmin(intensity[positive]) * 1e-3
        else:
            floor = 1e-99
        low = np.maximum(intensity - rms_error, floor)
        high = intensity + rms_error
        label = path.stem.replace(group + '_', '')
        ax.plot(radius[finite], intensity[finite], color=color, lw=2.0, label=label)
        ax.fill_between(radius[finite], low[finite], high[finite], color=color, alpha=0.16, lw=0)

    ax.set_title(group)
    ax.set_xlabel(r'$r\ [\mathrm{au}]$')
    ax.set_ylabel(r'$I_\lambda\ [\mathrm{Jy\ sr^{-1}}]$')
    ax.set_yscale('log')
    if all_positive:
        y = np.concatenate(all_positive)
        ax.set_ylim(np.nanmin(y) / 100.0, np.nanmax(y) * 10.0)
    ax.legend(frameon=False, ncols=min(3, len(paths)))
    ax.tick_params(direction='in', which='both', top=True, right=True)
    fig.tight_layout()
    plt.show()

## Opacity arrays

1D grid arrays are summarized first. The 2D opacity/scattering arrays are shown with color bars.

In [ ]:
opacity_dir = DATA / 'default_opacity_dsharp'
size_opac = np.load(opacity_dir / 'size_opac.npy')
lam_opac = np.load(opacity_dir / 'lam_opac.npy')
rhos_opac = np.load(opacity_dir / 'rhos_opac.npy')
k_abs_opac = np.load(opacity_dir / 'k_abs_opac.npy')
k_sca_opac = np.load(opacity_dir / 'k_sca_opac.npy')
g_sca_opac = np.load(opacity_dir / 'g_sca_opac.npy')

print('size_opac:', size_opac.shape, size_opac.min(), size_opac.max(), 'cm')
print('lam_opac :', lam_opac.shape, lam_opac.min(), lam_opac.max(), 'cm')
print('rhos_opac:', rhos_opac)
print('k_abs_opac:', k_abs_opac.shape)
print('k_sca_opac:', k_sca_opac.shape)
print('g_sca_opac:', g_sca_opac.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=150, constrained_layout=True)
axes[0].plot(size_opac, marker='.', lw=1.0)
axes[0].set_yscale('log')
axes[0].set_title('size_opac')
axes[0].set_xlabel('index')
axes[0].set_ylabel('grain radius [cm]')
axes[1].plot(lam_opac, marker='.', lw=1.0)
axes[1].set_yscale('log')
axes[1].set_title('lam_opac')
axes[1].set_xlabel('index')
axes[1].set_ylabel('wavelength [cm]')
for ax in axes:
    ax.tick_params(direction='in', which='both', top=True, right=True)
plt.show()

In [ ]:
fields = [
    ('k_abs_opac', k_abs_opac, True),
    ('k_sca_opac', k_sca_opac, True),
    ('g_sca_opac', g_sca_opac, False),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), dpi=150, constrained_layout=True)
for ax, (name, arr, log_scale) in zip(axes, fields):
    values = np.log10(np.maximum(arr, 1e-300)) if log_scale else arr
    mesh = ax.pcolormesh(lam_opac, size_opac, values, shading='auto', cmap='viridis')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\lambda\ [\mathrm{cm}]$')
    ax.set_ylabel(r'$a\ [\mathrm{cm}]$')
    ax.set_title((r'$\log_{10}$ ' if log_scale else '') + name)
    cb = fig.colorbar(mesh, ax=ax)
    cb.set_label((r'$\log_{10}$ ' if log_scale else '') + name)
    ax.tick_params(direction='in', which='both', top=True, right=True)
plt.show()